# Module 10.4: RLHF for Text-to-Image Generation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FlashVision/VisionRL/blob/main/Module_10_RL_For_Image_Generation/04_Text_To_Image_RL/text_to_image_rl.ipynb)

---

## Learning Objectives

By the end of this notebook, you will:

1. **Understand** the complete RLHF (Reinforcement Learning from Human Feedback) pipeline for image generation
2. **Derive** the mathematical formulation of reward model training, PPO, and KL penalty
3. **Know** how real systems (Stable Diffusion, DALL-E, InstructPix2Pix) apply RLHF
4. **Implement** a simplified RLHF pipeline: reward model + PPO-aligned generator
5. **Visualize** the effect of RLHF alignment on generation quality

---

## Prerequisites

- Understanding of policy optimization (PPO)
- Basic knowledge of generative models
- Familiarity with KL divergence

In [ ]:
!pip install torch torchvision numpy matplotlib scikit-image

In [ ]:
# Download REAL open-source dataset for image generation (TINY - under 200MB)
import torchvision
import torchvision.transforms as transforms

# MNIST for GAN training (classic, tiny, real)
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize([0.5], [0.5])])
mnist = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)
print(f"✅ MNIST: {len(mnist)} real images for GAN training (11MB)")

# FashionMNIST for more complex generation
fashion = torchvision.datasets.FashionMNIST(root='./data', train=True, download=True, transform=transform)
print(f"✅ FashionMNIST: {len(fashion)} real clothing images (30MB)")

# CIFAR-10 for color image generation
transform_color = transforms.Compose([transforms.ToTensor(), transforms.Normalize([0.5]*3, [0.5]*3)])
cifar10 = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform_color)
print(f"✅ CIFAR-10: {len(cifar10)} real color photos for generation (170MB)")

print("\n📦 Total download: ~211MB")
print("   NO CelebA needed - MNIST/FashionMNIST/CIFAR-10 are perfect for learning!")
print("   MNIST → simple GAN, FashionMNIST → medium, CIFAR-10 → color generation")

## Deep Derivation: Text-to-Image Generation with RL Alignment

### Step 1: CLIP Score as Text-Image Alignment Metric
CLIP embeds images and text in a shared space. The CLIP score:
$$\text{CLIP}(I, t) = \frac{f_I(I)^T f_T(t)}{\|f_I(I)\| \cdot \|f_T(t)\|} = \cos(f_I(I), f_T(t))$$

**Contrastive training objective (InfoNCE):**
$$\mathcal{L}_{\text{CLIP}} = -\frac{1}{N}\sum_{i=1}^N \left[\log\frac{\exp(\tau \cdot f_I(I_i)^T f_T(t_i))}{\sum_{j=1}^N \exp(\tau \cdot f_I(I_i)^T f_T(t_j))} + \log\frac{\exp(\tau \cdot f_T(t_i)^T f_I(I_i))}{\sum_{j=1}^N \exp(\tau \cdot f_T(t_i)^T f_I(I_j))}\right]$$

**Derivation:** This is a symmetric cross-entropy loss over a batch of $N$ image-text pairs. Each image should be most similar to its paired text (and vice versa) among all $N$ candidates.

The temperature $\tau$ is a learnable parameter. At convergence:
$$p(t_i | I_i) = \frac{\exp(\tau \cdot \text{sim}(I_i, t_i))}{\sum_j \exp(\tau \cdot \text{sim}(I_i, t_j))} \to 1 \quad \blacksquare$$

### Step 2: Diffusion Model Forward Process
The forward noising process gradually adds Gaussian noise:
$$q(x_t | x_{t-1}) = \mathcal{N}(x_t; \sqrt{1-\beta_t}\, x_{t-1}, \beta_t I)$$

**Closed-form for arbitrary $t$:** Using $\alpha_t = 1 - \beta_t$ and $\bar{\alpha}_t = \prod_{s=1}^t \alpha_s$:
$$q(x_t | x_0) = \mathcal{N}(x_t; \sqrt{\bar{\alpha}_t}\, x_0, (1-\bar{\alpha}_t) I)$$

**Derivation:** By induction. Base case ($t=1$): $x_1 = \sqrt{\alpha_1} x_0 + \sqrt{1-\alpha_1}\epsilon_1$.

Inductive step: Given $x_{t-1} = \sqrt{\bar{\alpha}_{t-1}} x_0 + \sqrt{1-\bar{\alpha}_{t-1}}\epsilon$:
$$x_t = \sqrt{\alpha_t} x_{t-1} + \sqrt{1-\alpha_t}\epsilon_t$$
$$= \sqrt{\alpha_t}(\sqrt{\bar{\alpha}_{t-1}} x_0 + \sqrt{1-\bar{\alpha}_{t-1}}\epsilon) + \sqrt{1-\alpha_t}\epsilon_t$$
$$= \sqrt{\bar{\alpha}_t} x_0 + \underbrace{\sqrt{\alpha_t(1-\bar{\alpha}_{t-1})}\epsilon + \sqrt{1-\alpha_t}\epsilon_t}_{\text{sum of independent Gaussians}}$$

The noise term has variance $\alpha_t(1-\bar{\alpha}_{t-1}) + (1-\alpha_t) = 1 - \bar{\alpha}_t$. $\blacksquare$

### Step 3: Denoising Score Matching
The reverse process learns $p_\theta(x_{t-1}|x_t) = \mathcal{N}(\mu_\theta(x_t, t), \sigma_t^2 I)$. The training objective:
$$\mathcal{L}_{\text{simple}} = E_{t, x_0, \epsilon}\left[\|\epsilon - \epsilon_\theta(x_t, t)\|^2\right]$$

**Connection to score matching:** The score function $\nabla_{x_t} \log q(x_t|x_0) = -\frac{\epsilon}{\sqrt{1-\bar{\alpha}_t}}$.

Therefore: $\epsilon_\theta(x_t, t) = -\sqrt{1-\bar{\alpha}_t} \cdot s_\theta(x_t, t)$ where $s_\theta \approx \nabla_{x_t} \log q(x_t)$.

**Derivation of equivalence to ELBO:** The full variational bound:
$$\mathcal{L}_{\text{VLB}} = E_q\left[-\log\frac{p_\theta(x_{0:T})}{q(x_{1:T}|x_0)}\right] = \underbrace{D_{KL}(q(x_T|x_0) \| p(x_T))}_{L_T} + \sum_{t>1}\underbrace{D_{KL}(q(x_{t-1}|x_t,x_0) \| p_\theta(x_{t-1}|x_t))}_{L_{t-1}} + \underbrace{(-\log p_\theta(x_0|x_1))}_{L_0}$$

The KL between Gaussians gives:
$$L_{t-1} = \frac{1}{2\sigma_t^2}\|\tilde{\mu}_t(x_t, x_0) - \mu_\theta(x_t, t)\|^2 + C$$

Substituting $\tilde{\mu}_t = \frac{1}{\sqrt{\alpha_t}}\left(x_t - \frac{\beta_t}{\sqrt{1-\bar{\alpha}_t}}\epsilon\right)$ yields $\mathcal{L}_{\text{simple}}$. $\blacksquare$

### Step 4: RLHF for Text-to-Image Models
Fine-tune diffusion model with human preference reward $R_\phi$:
$$J(\theta) = E_{x_0 \sim p_\theta(\cdot|t)}\left[R_\phi(x_0, t)\right] - \beta \cdot D_{KL}(p_\theta \| p_{\text{ref}})$$

The KL penalty prevents the model from deviating too far from the pretrained reference $p_{\text{ref}}$.

**Policy gradient through the denoising chain:** Since $x_0 = f_\theta(x_T, \epsilon_{1:T})$ is a deterministic function of noise (with DDIM):
$$\nabla_\theta J = E\left[\nabla_\theta R_\phi(f_\theta(x_T, \epsilon_{1:T}), t)\right]$$

This is the "ReFL" (Reward Feedback Learning) approach — backpropagate reward gradients through the denoising chain.

### Step 5: Reward Hacking and KL Regularization
Without KL constraint, the model exploits reward model weaknesses:
$$\max_\theta R_\phi(x) \quad \to \quad \text{adversarial examples for } R_\phi$$

**Derivation of optimal KL-constrained policy:**
$$\pi^*(x|t) = \frac{1}{Z(t)} \pi_{\text{ref}}(x|t) \exp\left(\frac{R_\phi(x,t)}{\beta}\right)$$

**Proof:** The Lagrangian is $L = E_\pi[R] - \beta D_{KL}(\pi \| \pi_{\text{ref}})$.
$$\frac{\delta L}{\delta \pi(x)} = R(x) - \beta \log\frac{\pi(x)}{\pi_{\text{ref}}(x)} - \beta - \lambda = 0$$
$$\log\frac{\pi^*(x)}{\pi_{\text{ref}}(x)} = \frac{R(x)}{\beta} - 1 - \frac{\lambda}{\beta}$$
$$\pi^*(x) = \pi_{\text{ref}}(x) \cdot \exp\left(\frac{R(x) - \beta - \lambda}{\beta}\right) = \frac{\pi_{\text{ref}}(x) \exp(R(x)/\beta)}{Z} \quad \blacksquare$$

### Step 6: DPO (Direct Preference Optimization) for Images
Instead of training a separate reward model, DPO directly optimizes from preferences:
$$\mathcal{L}_{\text{DPO}} = -E_{(x_w, x_l, t)}\left[\log\sigma\left(\beta\log\frac{p_\theta(x_w|t)}{p_{\text{ref}}(x_w|t)} - \beta\log\frac{p_\theta(x_l|t)}{p_{\text{ref}}(x_l|t)}\right)\right]$$

where $(x_w, x_l)$ are preferred/dispreferred image pairs.

**Derivation:** Starting from the Bradley-Terry preference model:
$$p(x_w \succ x_l | t) = \sigma(R(x_w, t) - R(x_l, t))$$

Substituting the optimal policy $R(x,t) = \beta\log\frac{\pi^*(x|t)}{\pi_{\text{ref}}(x|t)} + \beta\log Z(t)$:
$$p(x_w \succ x_l) = \sigma\left(\beta\log\frac{\pi^*(x_w|t)}{\pi_{\text{ref}}(x_w|t)} - \beta\log\frac{\pi^*(x_l|t)}{\pi_{\text{ref}}(x_l|t)}\right)$$

The $\log Z(t)$ terms cancel! MLE on this gives $\mathcal{L}_{\text{DPO}}$. $\blacksquare$

### Step 7: Classifier-Free Guidance (Mathematical Analysis)
At inference, combine conditional and unconditional scores:
$$\hat{\epsilon}_\theta(x_t, t, c) = \epsilon_\theta(x_t, t, \varnothing) + w \cdot (\epsilon_\theta(x_t, t, c) - \epsilon_\theta(x_t, t, \varnothing))$$

where $w > 1$ is the guidance scale.

**Derivation:** In score function form, this is:
$$\nabla_{x_t} \log \hat{p}(x_t|c) = \nabla_{x_t}\log p(x_t) + w \cdot \nabla_{x_t}\log p(c|x_t)$$

This corresponds to sampling from:
$$\hat{p}(x_t|c) \propto p(x_t) \cdot p(c|x_t)^w$$

For $w > 1$, we sharpen the conditional distribution, producing images with higher text-image alignment at the cost of diversity. The effective temperature is $1/w$. $\blacksquare$

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## 1. The RLHF Pipeline: Deep Mathematical Treatment

### 1.1 Overview

The RLHF pipeline for text-to-image consists of three stages:

1. **Pre-train** a generative model $G_\theta(x | c)$ on (text, image) pairs
2. **Train a reward model** $R_\phi(x, c)$ from human preference comparisons
3. **Fine-tune** $G_\theta$ using RL (PPO) to maximize $R_\phi$ while staying close to the pre-trained distribution

### 1.2 Stage 1: Pre-trained Generator

The base model $G_{\theta_0}$ is trained via maximum likelihood:

$$\theta_0 = \arg\max_\theta \; \mathbb{E}_{(x, c) \sim \mathcal{D}} \left[\log p_\theta(x | c)\right]$$

For diffusion models, this becomes the denoising score matching objective:

$$\mathcal{L}_{\text{DSM}} = \mathbb{E}_{x_0, \epsilon, t} \left[ \|\epsilon - \epsilon_\theta(x_t, t, c)\|^2 \right]$$

### 1.3 Stage 2: Reward Model Training

Given human preference data $\mathcal{D}_{\text{pref}} = \{(c, x_w, x_l)\}$ where $x_w \succ x_l$ ("$x_w$ is preferred over $x_l$ for prompt $c$"), the reward model is trained with the Bradley-Terry model:

$$P(x_w \succ x_l | c) = \sigma\left(R_\phi(x_w, c) - R_\phi(x_l, c)\right)$$

The loss function is:

$$\mathcal{L}_{\text{RM}}(\phi) = -\mathbb{E}_{(c, x_w, x_l) \sim \mathcal{D}_{\text{pref}}} \left[\log \sigma\left(R_\phi(x_w, c) - R_\phi(x_l, c)\right)\right]$$

### 1.4 Stage 3: PPO Fine-tuning

The RL objective maximizes expected reward with a KL penalty:

$$\max_\theta \; \mathbb{E}_{c \sim \mathcal{D}, x \sim G_\theta(\cdot|c)} \left[R_\phi(x, c)\right] - \beta \cdot \text{KL}\left[G_\theta(\cdot|c) \| G_{\theta_0}(\cdot|c)\right]$$

The KL penalty prevents **reward hacking** — the generator finding adversarial inputs that fool the reward model but look bad to humans.

### 1.5 PPO Objective (Clipped)

$$\mathcal{L}^{\text{CLIP}}(\theta) = \mathbb{E}_t \left[\min\left(\rho_t(\theta) \hat{A}_t, \; \text{clip}(\rho_t(\theta), 1-\epsilon, 1+\epsilon) \hat{A}_t\right)\right]$$

where $\rho_t(\theta) = \frac{\pi_\theta(a_t|s_t)}{\pi_{\theta_{\text{old}}}(a_t|s_t)}$ is the importance sampling ratio.

The advantage is computed as:
$$\hat{A}_t = R_\phi(x, c) - \beta \cdot \log \frac{G_\theta(x|c)}{G_{\theta_0}(x|c)} - V_\psi(c)$$

### 1.6 KL Divergence in Practice

For discrete token-based generators:
$$\text{KL}[G_\theta \| G_{\theta_0}] = \sum_x G_\theta(x|c) \log \frac{G_\theta(x|c)}{G_{\theta_0}(x|c)}$$

For continuous (e.g., diffusion) models, the KL is approximated per-sample:
$$\text{KL}_{\text{sample}} \approx \log G_\theta(x|c) - \log G_{\theta_0}(x|c)$$

## 2. How Real Systems Use RLHF

### DALL-E 3 / ChatGPT Image Generation
- Uses a reward model trained on human aesthetic ratings
- Fine-tunes the diffusion model with PPO on the denoising process
- KL penalty keeps images diverse and prevents mode collapse

### Stable Diffusion + DPO/RLHF
- **DDPO** (Denoising Diffusion Policy Optimization): treats the denoising chain as an MDP
  - State: $(x_t, t, c)$
  - Action: denoised estimate $x_{t-1}$
  - Reward: $R_\phi(x_0, c)$ at the final step
- **DPO for Diffusion**: Direct Preference Optimization avoids explicit reward model

### InstructPix2Pix
- Learns to follow editing instructions using paired before/after images
- RLHF ensures edits are faithful to instructions while preserving image quality

### The Alignment Tax
RLHF introduces a fundamental trade-off:

$$\underbrace{R_\phi(x, c)}_{\text{Human preference}} \quad \text{vs.} \quad \underbrace{\text{KL}[G_\theta \| G_{\theta_0}]}_{\text{Diversity preservation}}$$

Too little KL penalty $\to$ reward hacking, mode collapse. Too much $\to$ no improvement from RLHF.

## 3. Simplified Demo Setup

We build a toy "text-to-image" system:
- **Text** = class label (integer 0–4)
- **Image** = small 8×8 colored pattern
- Each class has a "target" pattern; the reward model learns to prefer images matching their class

In [ ]:
N_CLASSES = 5
IMG_SIZE = 8
NOISE_DIM = 32

def create_target_patterns():
    """Create distinct target patterns for each class."""
    patterns = {}
    
    # Class 0: red circle
    p = np.zeros((3, IMG_SIZE, IMG_SIZE), dtype=np.float32)
    y, x = np.ogrid[:IMG_SIZE, :IMG_SIZE]
    mask = ((x - 3.5)**2 + (y - 3.5)**2) < 9
    p[0][mask] = 0.9; p[1][mask] = 0.1; p[2][mask] = 0.1
    patterns[0] = p
    
    # Class 1: green horizontal stripes
    p = np.zeros((3, IMG_SIZE, IMG_SIZE), dtype=np.float32)
    for row in range(0, IMG_SIZE, 2):
        p[1, row, :] = 0.8
    patterns[1] = p
    
    # Class 2: blue diagonal
    p = np.zeros((3, IMG_SIZE, IMG_SIZE), dtype=np.float32)
    for i in range(IMG_SIZE):
        p[2, i, i] = 0.9
        if i > 0: p[2, i, i-1] = 0.5
        if i < IMG_SIZE-1: p[2, i, i+1] = 0.5
    patterns[2] = p
    
    # Class 3: yellow checkerboard
    p = np.zeros((3, IMG_SIZE, IMG_SIZE), dtype=np.float32)
    for row in range(IMG_SIZE):
        for col in range(IMG_SIZE):
            if (row + col) % 2 == 0:
                p[0, row, col] = 0.9
                p[1, row, col] = 0.9
    patterns[3] = p
    
    # Class 4: purple gradient
    p = np.zeros((3, IMG_SIZE, IMG_SIZE), dtype=np.float32)
    for col in range(IMG_SIZE):
        p[0, :, col] = col / IMG_SIZE * 0.8
        p[2, :, col] = 0.9 - col / IMG_SIZE * 0.5
    patterns[4] = p
    
    return patterns

target_patterns = create_target_patterns()

fig, axes = plt.subplots(1, N_CLASSES, figsize=(15, 3))
class_names = ['Red Circle', 'Green Stripes', 'Blue Diagonal', 'Yellow Checker', 'Purple Gradient']
for i, ax in enumerate(axes):
    ax.imshow(target_patterns[i].transpose(1, 2, 0), interpolation='nearest')
    ax.set_title(f'Class {i}: {class_names[i]}', fontsize=10, fontweight='bold')
    ax.axis('off')
plt.suptitle('Target Patterns for Each Class', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Generator (Pre-trained)

A small conditional generator that takes (noise, class_label) and outputs an 8x8 image.

In [ ]:
class ConditionalGenerator(nn.Module):
    """Simple conditional generator: (noise, class) -> 8x8 image."""
    
    def __init__(self, noise_dim=NOISE_DIM, n_classes=N_CLASSES, hidden_dim=128):
        super().__init__()
        self.class_embed = nn.Embedding(n_classes, 16)
        self.net = nn.Sequential(
            nn.Linear(noise_dim + 16, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 3 * IMG_SIZE * IMG_SIZE),
            nn.Sigmoid()
        )
    
    def forward(self, z, labels):
        c = self.class_embed(labels)
        x = torch.cat([z, c], dim=-1)
        out = self.net(x)
        return out.view(-1, 3, IMG_SIZE, IMG_SIZE)
    
    def log_prob(self, images, z, labels):
        """Approximate log probability under the generator."""
        generated = self.forward(z, labels)
        log_p = -F.mse_loss(generated, images, reduction='none').sum(dim=(1,2,3))
        return log_p


def pretrain_generator(gen, target_patterns, n_steps=1000, batch_size=64):
    """Pre-train the generator with supervised learning on target patterns."""
    optimizer = optim.Adam(gen.parameters(), lr=1e-3)
    
    for step in range(n_steps):
        labels = torch.randint(0, N_CLASSES, (batch_size,), device=device)
        z = torch.randn(batch_size, NOISE_DIM, device=device)
        
        targets = []
        for l in labels.cpu().numpy():
            noisy_target = target_patterns[l] + 0.15 * np.random.randn(3, IMG_SIZE, IMG_SIZE)
            targets.append(np.clip(noisy_target, 0, 1))
        targets = torch.FloatTensor(np.array(targets)).to(device)
        
        generated = gen(z, labels)
        loss = F.mse_loss(generated, targets)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        if (step + 1) % 250 == 0:
            print(f'  Pre-train step {step+1}/{n_steps}, Loss: {loss.item():.4f}')
    
    return gen

print('Pre-training generator...')
base_gen = ConditionalGenerator().to(device)
base_gen = pretrain_generator(base_gen, target_patterns)

ref_gen = ConditionalGenerator().to(device)
ref_gen.load_state_dict(base_gen.state_dict())
ref_gen.eval()
print('Pre-training complete. Reference model saved.')

In [ ]:
fig, axes = plt.subplots(2, N_CLASSES, figsize=(15, 6))

base_gen.eval()
with torch.no_grad():
    for i in range(N_CLASSES):
        z = torch.randn(1, NOISE_DIM, device=device)
        label = torch.tensor([i], device=device)
        img = base_gen(z, label).cpu().numpy()[0]
        
        axes[0, i].imshow(target_patterns[i].transpose(1, 2, 0), interpolation='nearest')
        axes[0, i].set_title(f'Target {i}', fontsize=10)
        axes[0, i].axis('off')
        
        axes[1, i].imshow(np.clip(img.transpose(1, 2, 0), 0, 1), interpolation='nearest')
        axes[1, i].set_title(f'Pre-trained {i}', fontsize=10)
        axes[1, i].axis('off')

base_gen.train()
plt.suptitle('Pre-trained Generator Outputs', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Reward Model Training

We train a reward model from synthetic preference pairs. The reward model scores how well an image matches its target class.

In [ ]:
class RewardModel(nn.Module):
    """Reward model: scores (image, class) pairs."""
    
    def __init__(self, n_classes=N_CLASSES, hidden_dim=128):
        super().__init__()
        self.class_embed = nn.Embedding(n_classes, 16)
        self.conv = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d(4),
            nn.Conv2d(16, 32, 3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d(2)
        )
        self.head = nn.Sequential(
            nn.Linear(32 * 4 + 16, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1)
        )
    
    def forward(self, images, labels):
        feat = self.conv(images).view(images.size(0), -1)
        c = self.class_embed(labels)
        x = torch.cat([feat, c], dim=-1)
        return self.head(x).squeeze(-1)


def generate_preference_data(gen, target_patterns, n_pairs=2000):
    """Generate synthetic preference pairs.
    
    For each pair: generate two images for the same class,
    prefer the one closer to the target pattern.
    """
    preferences = []
    gen.eval()
    
    with torch.no_grad():
        for _ in range(n_pairs):
            label = np.random.randint(N_CLASSES)
            label_t = torch.tensor([label, label], device=device)
            z = torch.randn(2, NOISE_DIM, device=device)
            imgs = gen(z, label_t).cpu().numpy()
            
            target = target_patterns[label]
            mse_0 = np.mean((imgs[0] - target) ** 2)
            mse_1 = np.mean((imgs[1] - target) ** 2)
            
            if mse_0 < mse_1:
                preferences.append((label, imgs[0], imgs[1]))  # imgs[0] preferred
            else:
                preferences.append((label, imgs[1], imgs[0]))  # imgs[1] preferred
    
    gen.train()
    return preferences


def train_reward_model(reward_model, preferences, n_epochs=15, batch_size=64):
    """Train reward model on preference pairs using Bradley-Terry loss."""
    optimizer = optim.Adam(reward_model.parameters(), lr=1e-3)
    losses = []
    accuracies = []
    
    for epoch in range(n_epochs):
        np.random.shuffle(preferences)
        epoch_loss = 0
        correct = 0
        total = 0
        
        for i in range(0, len(preferences) - batch_size, batch_size):
            batch = preferences[i:i + batch_size]
            labels = torch.tensor([b[0] for b in batch], device=device)
            winners = torch.FloatTensor(np.array([b[1] for b in batch])).to(device)
            losers = torch.FloatTensor(np.array([b[2] for b in batch])).to(device)
            
            r_win = reward_model(winners, labels)
            r_lose = reward_model(losers, labels)
            
            loss = -torch.log(torch.sigmoid(r_win - r_lose) + 1e-8).mean()
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            epoch_loss += loss.item()
            correct += (r_win > r_lose).float().sum().item()
            total += len(batch)
        
        avg_loss = epoch_loss / max(1, total // batch_size)
        accuracy = correct / max(1, total)
        losses.append(avg_loss)
        accuracies.append(accuracy)
        
        if (epoch + 1) % 3 == 0:
            print(f'  Epoch {epoch+1}/{n_epochs} | Loss: {avg_loss:.4f} | Accuracy: {accuracy:.3f}')
    
    return losses, accuracies


print('Generating preference data...')
preferences = generate_preference_data(base_gen, target_patterns, n_pairs=2000)
print(f'Generated {len(preferences)} preference pairs.')

print('\nTraining reward model...')
reward_model = RewardModel().to(device)
rm_losses, rm_accs = train_reward_model(reward_model, preferences)
print('Reward model training complete!')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(rm_losses, 'b-o', markersize=4)
axes[0].set_title('Reward Model Loss', fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Bradley-Terry Loss')
axes[0].grid(True, alpha=0.3)

axes[1].plot(rm_accs, 'g-o', markersize=4)
axes[1].set_title('Reward Model Preference Accuracy', fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].axhline(y=0.5, color='r', linestyle='--', alpha=0.5, label='Random chance')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('Reward Model Training', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 6. PPO Fine-tuning with KL Penalty

Now we fine-tune the generator using PPO to maximize the reward model's scores while penalizing deviation from the reference model.

In [ ]:
class ValueNetwork(nn.Module):
    """Value function V(c) for PPO baseline."""
    def __init__(self, n_classes=N_CLASSES, hidden_dim=64):
        super().__init__()
        self.embed = nn.Embedding(n_classes, 16)
        self.net = nn.Sequential(
            nn.Linear(16, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1)
        )
    
    def forward(self, labels):
        return self.net(self.embed(labels)).squeeze(-1)


def ppo_step(gen, ref_gen, reward_model, value_net,
             gen_optimizer, value_optimizer,
             batch_size=64, kl_coeff=0.1, clip_eps=0.2, n_ppo_epochs=4):
    """One PPO update step."""
    labels = torch.randint(0, N_CLASSES, (batch_size,), device=device)
    z = torch.randn(batch_size, NOISE_DIM, device=device)
    
    with torch.no_grad():
        old_images = gen(z, labels)
        old_log_probs = gen.log_prob(old_images, z, labels)
        ref_log_probs = ref_gen.log_prob(old_images, z, labels)
        
        rewards = reward_model(old_images, labels)
        kl = old_log_probs - ref_log_probs
        adjusted_rewards = rewards - kl_coeff * kl
        
        values = value_net(labels)
        advantages = adjusted_rewards - values
    
    total_policy_loss = 0
    total_value_loss = 0
    
    for _ in range(n_ppo_epochs):
        new_images = gen(z, labels)
        new_log_probs = gen.log_prob(new_images, z, labels)
        
        ratio = torch.exp(new_log_probs - old_log_probs.detach())
        surr1 = ratio * advantages.detach()
        surr2 = torch.clamp(ratio, 1 - clip_eps, 1 + clip_eps) * advantages.detach()
        policy_loss = -torch.min(surr1, surr2).mean()
        
        gen_optimizer.zero_grad()
        policy_loss.backward()
        nn.utils.clip_grad_norm_(gen.parameters(), 1.0)
        gen_optimizer.step()
        
        new_values = value_net(labels)
        value_loss = F.mse_loss(new_values, adjusted_rewards.detach())
        value_optimizer.zero_grad()
        value_loss.backward()
        value_optimizer.step()
        
        total_policy_loss += policy_loss.item()
        total_value_loss += value_loss.item()
    
    return {
        'policy_loss': total_policy_loss / n_ppo_epochs,
        'value_loss': total_value_loss / n_ppo_epochs,
        'mean_reward': rewards.mean().item(),
        'mean_kl': kl.mean().item(),
        'mean_adjusted_reward': adjusted_rewards.mean().item()
    }

print('PPO components defined.')

In [ ]:
rlhf_gen = ConditionalGenerator().to(device)
rlhf_gen.load_state_dict(base_gen.state_dict())

value_net = ValueNetwork().to(device)
gen_optimizer = optim.Adam(rlhf_gen.parameters(), lr=5e-5)
value_optimizer = optim.Adam(value_net.parameters(), lr=1e-3)

reward_model.eval()

n_ppo_steps = 150
ppo_logs = defaultdict(list)

print('Running PPO fine-tuning (RLHF)...')
for step in range(n_ppo_steps):
    metrics = ppo_step(
        rlhf_gen, ref_gen, reward_model, value_net,
        gen_optimizer, value_optimizer,
        batch_size=64, kl_coeff=0.1
    )
    
    for k, v in metrics.items():
        ppo_logs[k].append(v)
    
    if (step + 1) % 30 == 0:
        print(f'  Step {step+1}/{n_ppo_steps} | '
              f'Reward: {metrics["mean_reward"]:.3f} | '
              f'KL: {metrics["mean_kl"]:.3f} | '
              f'Adj Reward: {metrics["mean_adjusted_reward"]:.3f}')

print('PPO fine-tuning complete!')

## 7. Visualization: Before vs After RLHF

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

ax = axes[0, 0]
ax.plot(ppo_logs['mean_reward'], linewidth=2)
ax.set_title('Mean Reward (from Reward Model)', fontweight='bold')
ax.set_xlabel('PPO Step')
ax.set_ylabel('Reward')
ax.grid(True, alpha=0.3)

ax = axes[0, 1]
ax.plot(ppo_logs['mean_kl'], linewidth=2, color='orange')
ax.set_title('Mean KL Divergence (from Reference)', fontweight='bold')
ax.set_xlabel('PPO Step')
ax.set_ylabel('KL')
ax.grid(True, alpha=0.3)

ax = axes[1, 0]
ax.plot(ppo_logs['mean_adjusted_reward'], linewidth=2, color='green')
ax.set_title('Adjusted Reward (R - beta * KL)', fontweight='bold')
ax.set_xlabel('PPO Step')
ax.set_ylabel('Adjusted Reward')
ax.grid(True, alpha=0.3)

ax = axes[1, 1]
ax.plot(ppo_logs['policy_loss'], label='Policy Loss', linewidth=2)
ax.plot(ppo_logs['value_loss'], label='Value Loss', linewidth=2)
ax.set_title('PPO Losses', fontweight='bold')
ax.set_xlabel('PPO Step')
ax.set_ylabel('Loss')
ax.legend()
ax.grid(True, alpha=0.3)

plt.suptitle('RLHF Training Metrics', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(3, N_CLASSES, figsize=(15, 9))

base_gen.eval()
rlhf_gen.eval()

with torch.no_grad():
    for i in range(N_CLASSES):
        z = torch.randn(1, NOISE_DIM, device=device)
        label = torch.tensor([i], device=device)
        
        axes[0, i].imshow(target_patterns[i].transpose(1, 2, 0), interpolation='nearest')
        axes[0, i].set_title('Target', fontsize=10)
        axes[0, i].axis('off')
        
        pre_img = base_gen(z, label).cpu().numpy()[0]
        axes[1, i].imshow(np.clip(pre_img.transpose(1, 2, 0), 0, 1), interpolation='nearest')
        r_pre = reward_model(base_gen(z, label), label).item()
        axes[1, i].set_title(f'Pre-RLHF\nR={r_pre:.2f}', fontsize=9)
        axes[1, i].axis('off')
        
        post_img = rlhf_gen(z, label).cpu().numpy()[0]
        axes[2, i].imshow(np.clip(post_img.transpose(1, 2, 0), 0, 1), interpolation='nearest')
        r_post = reward_model(rlhf_gen(z, label), label).item()
        axes[2, i].set_title(f'Post-RLHF\nR={r_post:.2f}', fontsize=9)
        axes[2, i].axis('off')

axes[0, 0].set_ylabel('Target', fontsize=12, fontweight='bold')
axes[1, 0].set_ylabel('Pre-RLHF', fontsize=12, fontweight='bold')
axes[2, 0].set_ylabel('Post-RLHF', fontsize=12, fontweight='bold')

plt.suptitle('RLHF Alignment: Before vs After', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
n_samples = 50
pre_scores = []
post_scores = []

base_gen.eval()
rlhf_gen.eval()

with torch.no_grad():
    for c in range(N_CLASSES):
        labels = torch.full((n_samples,), c, dtype=torch.long, device=device)
        z = torch.randn(n_samples, NOISE_DIM, device=device)
        
        pre_imgs = base_gen(z, labels)
        post_imgs = rlhf_gen(z, labels)
        
        pre_r = reward_model(pre_imgs, labels).cpu().numpy()
        post_r = reward_model(post_imgs, labels).cpu().numpy()
        
        pre_scores.append(pre_r.mean())
        post_scores.append(post_r.mean())

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(N_CLASSES)
width = 0.35
ax.bar(x - width/2, pre_scores, width, label='Pre-RLHF', color='#3498db', alpha=0.8)
ax.bar(x + width/2, post_scores, width, label='Post-RLHF', color='#2ecc71', alpha=0.8)
ax.set_xlabel('Class', fontsize=12)
ax.set_ylabel('Mean Reward Score', fontsize=12)
ax.set_title('Reward Model Scores: Pre vs Post RLHF', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(class_names, rotation=15)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

## 8. Discussion: Limitations and Challenges

### Reward Hacking
Without the KL penalty, the generator may find **adversarial examples** that score highly on the reward model but look unnatural. The KL constraint keeps outputs close to the pre-trained distribution.

### Reward Model Quality
The RLHF pipeline is only as good as the reward model. In practice:
- Human preferences are noisy and inconsistent
- The reward model may not generalize beyond its training distribution
- Regularization and ensembling help mitigate overfitting

### Scaling Challenges

| Challenge | Solution in Practice |
|---|---|
| High-dimensional action space (diffusion steps) | DDPO treats denoising as a multi-step MDP |
| Expensive generation | Batch-wise RL updates, gradient accumulation |
| Unstable PPO on continuous outputs | Careful KL scheduling, low learning rates |
| Reward model distribution shift | Periodic reward model re-training |

### Beyond RLHF: Direct Preference Optimization (DPO)

DPO eliminates the explicit reward model by directly optimizing:

$$\mathcal{L}_{\text{DPO}}(\theta) = -\mathbb{E}_{(c, x_w, x_l)}\left[\log \sigma\left(\beta \log \frac{\pi_\theta(x_w|c)}{\pi_{\text{ref}}(x_w|c)} - \beta \log \frac{\pi_\theta(x_l|c)}{\pi_{\text{ref}}(x_l|c)}\right)\right]$$

This is equivalent to RLHF with the optimal reward model, but simpler to implement.

## Summary

In this notebook, we explored the RLHF pipeline for text-to-image generation:

1. **Three-stage pipeline**: Pre-train generator, train reward model from preferences, fine-tune with PPO

2. **Reward model**: Learns from pairwise comparisons using the Bradley-Terry model, enabling the system to capture nuanced human preferences

3. **PPO with KL penalty**: Fine-tunes the generator to maximize reward while preventing reward hacking through a KL divergence constraint against the reference model

4. **Real-world applications**: DALL-E 3, Stable Diffusion (DDPO), and InstructPix2Pix all use variants of this pipeline to align image generation with human preferences

5. **Key trade-off**: The KL coefficient $\beta$ controls the alignment tax — too low allows reward hacking, too high prevents meaningful improvement

### Next Steps

- Implement DPO as an alternative to PPO
- Scale to real diffusion models with DDPO
- Explore multi-objective RLHF (safety + quality + faithfulness)